# Hubbard–Holstein Trotter-step sweep

Cleaned and organized for reproducible execution from the repository root.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from qutip import (
    tensor, basis, destroy, qeye,
    sigmaz, sigmax, sigmay,
    propagator, fidelity
)


In [ ]:
# 系统参数
k = 1.0
U = 8.0 * k
g = 0.1 * k
omega0 = 8.0 * k
n_boson = 8
T_total = 1.0


In [ ]:
# 构造算符函数与全局算符（与 N_trot 无关）
def tensor_ops(ops):
    out = ops[0]
    for o in ops[1:]: out = tensor(out, o)
    return out


In [ ]:
# Boson 模式
a1 = tensor(destroy(n_boson), qeye(n_boson), qeye(2), qeye(2), qeye(2), qeye(2))
a2 = tensor(qeye(n_boson), destroy(n_boson), qeye(2), qeye(2), qeye(2), qeye(2))
# 自旋 σ_z 列表
q_z = []
for i in range(4):
    ops = [qeye(n_boson), qeye(n_boson)] + [sigmaz() if j==i else qeye(2) for j in range(4)]
    q_z.append(tensor_ops(ops))

In [ ]:
# Hamiltonian 各项
ee = U/4 * ((q_z[0]+q_z[1]) + q_z[0]*q_z[1] + (q_z[2]+q_z[3]) + q_z[2]*q_z[3])
eb = g/2 * ((q_z[0]+q_z[1])*(a1.dag()+a1) + (q_z[2]+q_z[3])*(a2.dag()+a2))
hop = 0
for i,j in [(1,2),(0,3)]:
    hop += -k * tensor_ops([qeye(n_boson), qeye(n_boson)] + [sigmax() if idx in (i,j) else qeye(2) for idx in range(4)])
    hop += -k * tensor_ops([qeye(n_boson), qeye(n_boson)] + [sigmay() if idx in (i,j) else qeye(2) for idx in range(4)])


In [ ]:
# 初始态 & 全局常量态
ferm = tensor(basis(2,0), basis(2,1), basis(2,1), basis(2,0))
boson = tensor(basis(n_boson,0), basis(n_boson,0))
psi0 = tensor(boson, ferm)


In [ ]:
# 不同 Trotter 步数集合
N_list = [50, 100, 200, 400]
results = {}

tick = plt.ticklabel_format  # 关闭偏移提示

In [ ]:
# 针对每个 N 进行仿真并收集保真度曲线
for N in N_list:
    dt = T_total / N
    # 预计算指数算符
    H_analog = omega0*(a1.dag()*a1 + a2.dag()*a2) + ee + eb
    H_hop = hop
    U_analog  = (-1j * H_analog * dt).expm()
    U_digital = (-1j * H_hop    * dt).expm()
    # 精确演化
    times = np.linspace(0, T_total, N+1)
    U_ex = [propagator(H_analog+H_hop, t) for t in times]
    psi_ex = [U_ex[i]*psi0 for i in range(N+1)]
    # Trotter 演化
    psi = psi0
    fid = [1.0]
    for i in range(1, N+1):
        psi = U_digital * (U_analog * psi)
        fid.append(fidelity(psi_ex[i], psi))
    results[N] = (times, fid)

In [ ]:
# 绘制多条曲线
plt.figure()
tick(useOffset=False, style='plain')
for N, (t, f) in results.items():
    plt.plot(t, f, marker='o', label=f'N={N}')
plt.xlabel('Time (1/k)')
plt.ylabel('Fidelity')
plt.title('DAQC Fidelity vs Time for Various N_trot')
plt.legend()
plt.show()